In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Importar librerías para la red neuronal
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

# Cargar los datos
url = 'https://github.com/adiacla/bigdata/raw/master/riesgo.xlsx'
df = pd.read_excel(url)

# Mostrar las primeras filas y la información general
print(df.head())
print(df.info())

In [ ]:
# --- Análisis de la variable objetivo ---
print(df['Credit_Score'].value_counts())
sns.countplot(x='Credit_Score', data=df)
plt.title('Distribución de Credit Score')
plt.show()

# --- Identificar y eliminar columnas irrelevantes ---
columnas_a_eliminar = ['SSN', 'Customer_ID', 'Name'] # Añade otras si las hay
df_limpio = df.drop(columns=[col for col in columnas_a_eliminar if col in df.columns])

# --- Manejo de valores nulos (ejemplo simple) ---
print(df_limpio.isnull().sum())
# Estrategia: rellenar numéricas con la mediana y categóricas con la moda
for col in df_limpio.columns:
    if df_limpio[col].dtype == 'object' and col != 'Credit_Score':
        df_limpio[col].fillna(df_limpio[col].mode()[0], inplace=True)
    elif df_limpio[col].dtype in ['int64', 'float64']:
        df_limpio[col].fillna(df_limpio[col].median(), inplace=True)

# --- Separar características (X) y variable objetivo (y) ---
X = df_limpio.drop('Credit_Score', axis=1)
y = df_limpio['Credit_Score']

# Codificar la variable objetivo si está en texto (ej: 'Good', 'Standard', 'Poor')
# Asumimos que ya son números (0,1,2). Si no:
# label_encoder = LabelEncoder()
# y = label_encoder.fit_transform(y)

In [ ]:
# --- Identificar columnas numéricas y categóricas ---
columnas_numericas = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
columnas_categoricas = X.select_dtypes(include=['object']).columns.tolist()

print(f"Columnas numéricas: {columnas_numericas}")
print(f"Columnas categóricas: {columnas_categoricas}")

# --- Transformadores de Sklearn ---
# Para numéricas: Imputar (si quedaron nulos) y escalar
# Usaremos SimpleImputer por si acaso, aunque ya imputamos arriba
from sklearn.impute import SimpleImputer
transformador_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Por si acaso
    ('scaler', StandardScaler())
])

# Para categóricas: Imputar y One-Hot Encoding
transformador_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# --- Combinar transformadores en un ColumnTransformer ---
preprocesador = ColumnTransformer(
    transformers=[
        ('num', transformador_numerico, columnas_numericas),
        ('cat', transformador_categorico, columnas_categoricas)
    ])

# --- Añadir PCA al pipeline final ---
# Primero transformamos los datos para saber cuántas componentes elegir
X_preprocesado = preprocesador.fit_transform(X)
print(f"Dimensiones después del preprocesamiento: {X_preprocesado.shape[1]}")

# Calcular varianza explicada para elegir n_components
pca_prueba = PCA().fit(X_preprocesado)
plt.plot(np.cumsum(pca_prueba.explained_variance_ratio_))
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Acumulada Explicada')
plt.grid()
plt.show()
# Elige un número, por ejemplo, 20 si explica > 90% de la varianza

n_componentes_pca = 20 # Ajusta este número según el gráfico
pipeline_completo = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('pca', PCA(n_components=n_componentes_pca))
])

# --- Dividir datos en entrenamiento y prueba ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Aplicar el pipeline a los datos de entrenamiento (fit y transform)
X_train_proc = pipeline_completo.fit_transform(X_train)
# Solo transformar a los datos de prueba
X_test_proc = pipeline_completo.transform(X_test)

print(f"Dimensiones de entrenamiento después de PCA: {X_train_proc.shape}")

In [ ]:
# --- Definir el modelo ---
modelo = models.Sequential(name="ANN_Credit_Score")

# Capa de entrada (el tamaño es el número de componentes PCA)
modelo.add(layers.Input(shape=(X_train_proc.shape[1],)))

# Capas ocultas
modelo.add(layers.Dense(128, activation='relu'))
modelo.add(layers.Dropout(0.3)) # Para regularización
modelo.add(layers.Dense(64, activation='relu'))
modelo.add(layers.Dropout(0.3))
modelo.add(layers.Dense(32, activation='relu'))

# Capa de salida: 3 neuronas (una por clase) con activación 'softmax'
modelo.add(layers.Dense(3, activation='softmax'))

# Mostrar resumen del modelo
modelo.summary()

# --- Compilar el modelo ---
modelo.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy', # Porque las etiquetas son enteros (0,1,2)
              metrics=['accuracy'])

# --- Entrenar el modelo ---
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

historial = modelo.fit(
    X_train_proc, y_train,
    validation_split=0.2, # Usar 20% de entrenamiento para validación durante el entrenamiento
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# --- Graficar la evolución del entrenamiento ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(historial.history['loss'], label='Pérdida (train)')
plt.plot(historial.history['val_loss'], label='Pérdida (validación)')
plt.legend()
plt.title('Evolución de la Pérdida')

plt.subplot(1, 2, 2)
plt.plot(historial.history['accuracy'], label='Exactitud (train)')
plt.plot(historial.history['val_accuracy'], label='Exactitud (validación)')
plt.legend()
plt.title('Evolución de la Exactitud')
plt.show()

In [ ]:
# --- Evaluar en el conjunto de prueba ---
loss, accuracy = modelo.evaluate(X_test_proc, y_test, verbose=0)
print(f"Pérdida en prueba: {loss:.4f}")
print(f"Exactitud en prueba: {accuracy:.4f}")

# --- Obtener predicciones ---
y_pred_proba = modelo.predict(X_test_proc)
y_pred = np.argmax(y_pred_proba, axis=1)

# --- Matriz de Confusión ---
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de Confusión')
plt.show()

# --- Reporte de Clasificación ---
print(classification_report(y_test, y_pred, target_names=['Score 0', 'Score 1', 'Score 2']))